# 🚀 Quickstart — all 13 domains — One-click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/quickstart_colab.ipynb)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Task](https://img.shields.io/badge/task-seg%20%2B%20cd-4c8cbf) ![Auto](https://img.shields.io/badge/runs-automatically-3fb950)

The 5-minute tour: pick **any of the 13 disaster domains**, choose segmentation or change detection, upload your scene (with inline preview) or fall back to a labeled synthetic sample — results, dashboard and export happen automatically.

| ▶️ How to run | |
|---|---|
| **Everything automatic** | Click **Runtime ▸ Run all** (Ctrl+F9): setup → demo scene → inference → dashboard → export chain without any further clicks |
| **With your data** | Run the cells once, then click the 📤 upload buttons — the pipeline re-runs automatically after upload |
| **No image at hand?** | The 🛟 fallback generates a clearly-labeled synthetic flood scene — the notebook *never blocks or crashes* |

**📤 Upload preview — what to have ready:**

| file | what it is | typical size |
|---|---|---|
| **`your_scene.png`** | any domain, optical or SAR | 512–2048 px, ≤ 20 MB |
| **`pre + post pair`** | for change-detection mode | same area |
| **`no image?`** | 🛟 synthetic sample for the chosen domain | — |

> 🔖 Every result is stamped with its mode: **CHECKPOINT** (your `.pth`), **QUICK-TRAIN** (genuinely briefly-trained), or **SYNTHETIC-DEMO** (clearly-labeled heuristic). Nothing is faked.

In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent, restart-safe {display-mode:"form"}
AUTO_RUN = True  #@param {type:"boolean"}
SHOW_PREVIEW_ON_UPLOAD = True  #@param {type:"boolean"}
import os, sys, glob, json, math, random, subprocess, traceback, datetime, textwrap

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

colab_kit.pip_install("ipywidgets", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from colab_kit import (make_synthetic, infer, overlay, heatmap, to_u8, stats_for,
                       stylize_poster, read_any, verdict, show_pair, run_inference,
                       run_dashboard, export_results_zip_a, nb_save_upload, nb_apply,
                       nb_upload_pre, nb_upload_post, nb_use_synthetic,
                       _field, _fbm, _stretch, _blur, _median_mask, _terrain, _hsv,
                       segment_heuristic, change_heuristic, _otsu)

random.seed(42); np.random.seed(42)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
DOMAIN = "flood"          # this notebook's disaster domain
TASK   = "both"            # "seg" | "cd" | "both"
GSD_M  = 10.0               # ground sample distance (m/pixel) for km² stats
NOTEBOOK = "quickstart_colab.ipynb"
STATE = {"domain": DOMAIN, "task": "seg" if TASK != "cd" else "cd",
         "pre": None, "post": None, "gt": None, "mask": None, "source": "none"}
print("✅ Environment ready — notebook:", NOTEBOOK, "| domain:", DOMAIN, "| task:", TASK)

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — buttons + inline preview, never blocks {display-mode:"form"}
# upload logic lives in colab_kit (single source of truth, unit-tested in CI)
def _save_upload(name, data):
    return colab_kit.nb_save_upload(globals(), name, data)

def _apply(kind, arr):
    return colab_kit.nb_apply(globals(), kind, arr)

def upload_pre(_=None):
    return colab_kit.nb_upload_pre(globals())

def upload_post(_=None):
    return colab_kit.nb_upload_post(globals())

def use_synthetic(_=None):
    return colab_kit.nb_use_synthetic(globals())


SPEC = [['your_scene.png', 'any domain, optical or SAR', '512–2048 px, ≤ 20 MB'],
 ['pre + post pair', 'for change-detection mode', 'same area'],
 ['no image?', '🛟 synthetic sample for the chosen domain', '—']]
W = 70
def print_spec():
    print("┌" + "─" * W + "┐")
    for line in textwrap.wrap("📤 WHAT TO UPLOAD — " + NOTEBOOK, W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    for label, desc, size in SPEC:
        wrapped = textwrap.wrap(label + " — " + desc + "  (" + size + ")", W - 6)
        for j, line in enumerate(wrapped):
            print(("│   • " if j == 0 else "│     ") + line.ljust(W - 6) + "│")
    for line in textwrap.wrap(
            "accepted: .png .jpg .jpeg .tif .tiff .h5 | sweet spot: 512–2048 px, ≤ 20 MB each | "
            "GeoTIFF georeference is kept when present", W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    print("└" + "─" * W + "┘")
print_spec()

out_up = widgets.Output(layout={"border": "1px solid #ddd"})
domain_sel = widgets.Dropdown(
    options=["flood", "landslide", "wildfire", "oil_spill", "avalanche",
             "volcano", "glof", "dust_storm", "smoke_plume", "locust",
             "tsunami", "hurricane", "infra_damage"],
    value="flood", description="Domain:")
def _sync_dom(*_):
    STATE["domain"] = domain_sel.value
domain_sel.observe(_sync_dom, names="value")
task_sel = widgets.ToggleButtons(
    options=[("🖼️ single image (segmentation)", "seg"),
             ("🆚 pre/post pair (change detection)", "cd")],
    value="seg", description="Task:")
def _sync_task(*_):
    STATE["task"] = task_sel.value
    btn_post.layout.display = None if task_sel.value == "cd" else "none"
task_sel.observe(_sync_task, names="value")
gsd_box = widgets.FloatText(value=GSD_M, description="GSD m/px:",
                            tooltip="ground sample distance, used for km² statistics")
btn_post = widgets.Button(description="⬆️ Upload POST image", button_style="success", icon="upload")
btn_extra = None
# (no extra upload buttons)

btn_pre = widgets.Button(description="⬆️ Upload PRE image", button_style="primary", icon="upload")
btn_synth = widgets.Button(description="🛟 Use bundled synthetic sample",
                           button_style="warning", icon="picture")
btns = [btn_pre]
if TASK in ("cd", "both"):
    btns.insert(1, btn_post)
if btn_extra is not None:
    btns.append(btn_extra)
box = widgets.VBox([widgets.HBox(btns), widgets.HBox([btn_synth, gsd_box]), out_up])






btn_pre.on_click(upload_pre)
if TASK in ("cd", "both"):
    btn_post.on_click(upload_post)
btn_synth.on_click(use_synthetic)
display(box)

if AUTO_RUN and STATE["pre"] is None:
    print("⚡ AUTO_RUN: generating bundled synthetic sample — upload later to re-run automatically")
    use_synthetic()

In [ ]:
#@title ▶️ RUN — labeled inference + results dashboard {display-mode:"form"}
MODE_TAG = "SYNTHETIC-DEMO"

def try_checkpoint():
    """No checkpoint mode in this notebook — heuristic demo only."""
    return None

def run_inference():
    return colab_kit.run_inference(globals(), try_checkpoint)

def run_pipeline():
    return colab_kit.run_dashboard(globals(), try_checkpoint)

run_pipeline() if AUTO_RUN else print("ready — call run_pipeline() after uploading")

In [ ]:
#@title 📦 EXPORT — results zip + optional Drive copy + auto-download {{display-mode:"form"}}
USE_DRIVE = False  #@param {{type:"boolean"}}

def run_export():
    return colab_kit.export_results_zip_a(globals(), use_drive=USE_DRIVE)

run_export() if AUTO_RUN else print("call run_export() when ready")

## 🗺️ Everything this collection covers

| disaster domain | notebook |
|---|---|
| 💧 Flood | [`flood_colab.ipynb`](flood_colab.ipynb) |
| 🔥 Wildfire | [`wildfire_colab.ipynb`](wildfire_colab.ipynb) |
| 🌊 Tsunami | [`tsunami_colab.ipynb`](tsunami_colab.ipynb) |
| 🌀 Hurricane | [`hurricane_colab.ipynb`](hurricane_colab.ipynb) |
| ⛰️ Landslide | [`landslide_colab.ipynb`](landslide_colab.ipynb) |
| 🌋 Volcano | [`volcano_colab.ipynb`](volcano_colab.ipynb) |
| 🛢️ Oil spill · 🏔️ Avalanche · 🧊 GLOF · 🏜️ Dust · 💨 Smoke · 🦗 Locust · 🏚️ Infra damage | use the Domain dropdown in THIS notebook, or [`batch_inference_colab.ipynb`](batch_inference_colab.ipynb) |
| 🆚 Change detection (BIT/ChangeOS) | [`change_detection_colab.ipynb`](change_detection_colab.ipynb) |
| 🛰️ Fine-tune on your data | [`satellite_finetune_colab.ipynb`](satellite_finetune_colab.ipynb) |
| 🎨 AI illustration images | [`ai_images_colab.ipynb`](ai_images_colab.ipynb) |
| 🎞️ README demo GIF | [`demo_gif_colab.ipynb`](demo_gif_colab.ipynb) |
| 📦 Batch processing | [`batch_inference_colab.ipynb`](batch_inference_colab.ipynb) |
| 🌐 Gradio web demo | [`gradio_demo/`](gradio_demo/) |

> 🔒 Synthetic fallback scenes are procedural AI-style illustrations — no real photos anywhere.